## Gustavo Hernández Angeles
### Neural Language Model by Bengio

In [1]:
import os
import time
import shutil
import random
from typing import Literal, Tuple
from argparse import Namespace
import matplotlib.pyplot as plt

# Preprocessing
import nltk
from nltk.corpus import stopwords
from nltk import ngrams
from nltk.tokenize import TweetTokenizer
from nltk import FreqDist
import pandas as pd
import numpy as np

# PyTorch
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.nn.functional as F

# Sci-kit Learn
from sklearn.metrics import accuracy_score

In [2]:
seed = 1111
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.backends.cudnn.benchmark = False

In [3]:
X_train = pd.read_csv("./data/mex_train.txt", 
                      sep ="\r\n", engine='python', header=None)[0].tolist()
X_val = pd.read_csv("./data/mex_val.txt", sep ="\r\n", engine='python'
                    , header=None)[0].tolist()

In [4]:
args = Namespace() # espacio de parámetros
args.N = 4 # Tetra-gramas

In [5]:
# Creamos la clase de n-gramas
class NgramData:
    def __init__(self, N: int, vocab_max: int = 5000,
                 tokenizer=None, embeddings_model=None):
        """
        Clase para crear n-gramas a partir de un texto
        Args:
            N (int): Número de n-gramas
            vocab_max (int): Tamaño máximo del vocabulario
            tokenizer (callable): Tokenizador
            embeddings_model (np.ndarray): Matriz de embeddings
        """
        self.N = N
        self.vocab_max = vocab_max
        self.tokenizer = tokenizer if tokenizer else self.default_tokenizer
        self.punct = set([".", ",", "!", "?", "¿", "¡", ";", ":", "...", "(",
                          ")","[", "]", "{", "}", ">", "^", "<", "<url>", "*",
                          "@usuario"])
        self.UNK = "<unk>"
        self.SOS = "<s>"
        self.EOS = "</s>"
        self.embeddings_model = embeddings_model

    def get_vocab_size(self) -> int:
        return len(self.vocab)

    def default_tokenizer(self, text: str) -> list:
        return text.split(" ")

    def fit(self, X: list) -> None:
        self.vocab = self.get_vocab(X)
        self.vocab.add(self.UNK)
        self.vocab.add(self.SOS)
        self.vocab.add(self.EOS)
        
        # Diccionario de palabras a índices e índices a palabras
        self.w2id = {}
        self.id2w = {}
        
        if self.embeddings_model is not None:
            self.embeddings_matrix = np.empty(
                (len(self.vocab), self.embeddings_model.shape[1])
            )
        
        id = 0
        for doc in X:
            for word in self.tokenizer(doc):
                word_ = word.lower()
                if word_ in self.vocab and not word_ in self.w2id:
                    self.w2id[word_] = id
                    self.id2w[id] = word_
                    if self.embeddings_model is not None:
                        if word_ in self.embeddings_model:
                            self.embeddings_matrix[id] = \
                                self.embeddings_model[word_]
                        else:
                            self.embeddings_matrix[id] = \
                                np.random.rand(
                                    self.embeddings_matrix.vector_size
                                    )
                    id += 1
        
        self.w2id.update({self.UNK: id, self.SOS: id+1, self.EOS: id+2})
        self.id2w.update({id: self.UNK, id+1: self.SOS, id+2: self.EOS})
        
    def transform(self, X: list) -> np.ndarray:
        X_ngrams = []
        y = []
        
        for doc in X:
            doc_ngram = self.get_ngram_doc(doc)
            for words_window in doc_ngram:
                # Convertimos las palabras a índices
                words_window_ids = [self.w2id[w] 
                                    for w in words_window] 
                # Tomamos todas las palabras menos la última
                X_ngrams.append(list(words_window_ids[:-1])) 
                # Tomamos la última palabra
                y.append(words_window_ids[-1]) 
        return np.array(X_ngrams), np.array(y)
        
    def get_ngram_doc(self, doc: str) -> list:
        doc_tokens = self.tokenizer(doc)
        doc_tokens = self.replace_unk(doc_tokens)
        doc_tokens = [w.lower() for w in doc_tokens]
        doc_tokens = [self.SOS] * (self.N-1) + doc_tokens + [self.EOS]
        return list(ngrams(doc_tokens, self.N))
        
    def replace_unk(self, doc_tokens: list) -> list:
        for i, token in enumerate(doc_tokens):
            if token.lower() not in self.vocab:
                doc_tokens[i] = self.UNK
        return doc_tokens
        
    def get_vocab(self, X: list):
        freqdist = FreqDist([word.lower() for sentence in X
                             for word in self.tokenizer(sentence)
                             if not self.remove_word(word)])
        sorted_words = self.sortFreqDist(
            freqdist)[:self.vocab_max-3]  # Por los tokens especiales
        return set(sorted_words)

    def remove_word(self, word: str) -> bool:
        word = word.lower()
        is_punct = True if word in self.punct else False
        is_digit = word.isnumeric()
        return is_punct or is_digit

    def sortFreqDist(self, freqdist: FreqDist) -> list:
        freq_dict = dict(freqdist)
        return sorted(freq_dict, key=freq_dict.get, reverse=True)

In [6]:
tk = TweetTokenizer()
ngram_data = NgramData(args.N, 5000, tk.tokenize)
ngram_data.fit(X_train)

In [7]:
print("Vocab Size: ", ngram_data.get_vocab_size())

Vocab Size:  5000


In [8]:
"""
'lo peor de todo es que no me acuerdo de si me tome la pastilla 
de la tension o no'

[['<s>', '<s>', '<s>'],
 ['<s>', '<s>', 'lo'],
 ['<s>', 'lo', 'peor'],
 ['lo', 'peor', 'de'],
 ['peor', 'de', 'todo'],
 ['de', 'todo', 'es'],
 ['todo', 'es', 'que'],
 ['es', 'que', 'no'],
 ['que', 'no', 'me'],
 ['no', 'me', 'acuerdo'],
 ['me', 'acuerdo', 'de'],
 ['acuerdo', 'de', 'si'],
 ['de', 'si', 'me'],
 ['si', 'me', 'tome'],
 ['me', 'tome', 'la'],
 ['tome', 'la', 'pastilla'],
 ['la', 'pastilla', 'de'],
 ['pastilla', 'de', 'la'],
 ['de', 'la', 'tension'],
 ['la', 'tension', 'o'],
 ['tension', 'o', 'no'],
 ['o', 'no', '</s>'],
 ['no', '</s>', '</s>']]
"""

"\n'lo peor de todo es que no me acuerdo de si me tome la pastilla \nde la tension o no'\n\n[['<s>', '<s>', '<s>'],\n ['<s>', '<s>', 'lo'],\n ['<s>', 'lo', 'peor'],\n ['lo', 'peor', 'de'],\n ['peor', 'de', 'todo'],\n ['de', 'todo', 'es'],\n ['todo', 'es', 'que'],\n ['es', 'que', 'no'],\n ['que', 'no', 'me'],\n ['no', 'me', 'acuerdo'],\n ['me', 'acuerdo', 'de'],\n ['acuerdo', 'de', 'si'],\n ['de', 'si', 'me'],\n ['si', 'me', 'tome'],\n ['me', 'tome', 'la'],\n ['tome', 'la', 'pastilla'],\n ['la', 'pastilla', 'de'],\n ['pastilla', 'de', 'la'],\n ['de', 'la', 'tension'],\n ['la', 'tension', 'o'],\n ['tension', 'o', 'no'],\n ['o', 'no', '</s>'],\n ['no', '</s>', '</s>']]\n"

In [9]:
X_ngram_train, y_ngram_train = ngram_data.transform(X_train)
X_ngram_val, y_ngram_val = ngram_data.transform(X_val)

In [10]:
print(f"Training observations: {X_ngram_train.shape[0]}")
print(f"Validation observations: {X_ngram_val.shape[0]}")

Training observations: 102751
Validation observations: 11558


In [11]:
[[ngram_data.id2w[w] for w in ngram] for ngram in X_ngram_train[:10]]

[['<s>', '<s>', '<s>'],
 ['<s>', '<s>', '<unk>'],
 ['<s>', '<unk>', '<unk>'],
 ['<unk>', '<unk>', '<unk>'],
 ['<unk>', '<unk>', 'q'],
 ['<unk>', 'q', 'se'],
 ['q', 'se', 'puede'],
 ['se', 'puede', 'esperar'],
 ['puede', 'esperar', 'del'],
 ['esperar', 'del', 'maricon']]

In [46]:
args.batch_size = 512

args.num_workers = 2


#Train
train_dataset = TensorDataset(torch.tensor(X_ngram_train, dtype=torch.long),
                              torch.tensor(y_ngram_train, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=args.batch_size,
                          num_workers=args.num_workers, shuffle=True)

#Validation

val_dataset = TensorDataset(torch.tensor(X_ngram_val, dtype=torch.long),
                            torch.tensor(y_ngram_val, dtype=torch.long))

val_loader = DataLoader(val_dataset, batch_size=args.batch_size,
                        num_workers=args.num_workers, shuffle=False)

In [36]:
batch = next(iter(train_loader))
print(f"X shape: {batch[0].shape}") # Tensor con las features
print(f"y shape: {batch[1].shape}") # Tensor con las etiquetas

X shape: torch.Size([128, 3])
y shape: torch.Size([128])


In [37]:
# Vocab size
args.vocab_size = ngram_data.get_vocab_size()

# Embedding size
args.embedding_dim = 50

# Dimensions for hidden layer
args.d_h = 100

args.dropout = 0.1

In [38]:
class NeuralLM(nn.Module):
    
    def __init__(self, args):
        super(NeuralLM, self).__init__()
        
        self.window_size = args.N - 1
        self.embedding_size = args.embedding_dim
        
        self.emb = nn.Embedding(args.vocab_size, args.embedding_dim)
        self.fc1 = nn.Linear(args.embedding_dim * self.window_size, args.d_h)
        # De manera aleatoria, apagamos un porcentaje de neuronas,
        # evitando el overfitting
        self.drop1 = nn.Dropout(p=args.dropout)
        self.fc2 = nn.Linear(args.d_h, args.vocab_size, bias=False)
        
    def forward(self, x):
        x = self.emb(x)
        x = x.view(-1, self.window_size*self.embedding_size)
        h = F.relu(self.fc1(x)) # relu(x) = max(0, x)
        h = self.drop1(h) # Darle sapes a la red
        return self.fc2(h)
        

In [39]:
e = nn.Embedding(10, 3)
e.weight

Parameter containing:
tensor([[ 0.1819,  1.2492,  1.3321],
        [-0.9755, -2.3504, -1.5249],
        [ 0.7728, -0.0667,  0.2542],
        [-1.6045, -0.2331,  1.1437],
        [-0.1892, -2.4593,  0.2732],
        [-0.2605,  1.5861,  0.6683],
        [-0.4287, -0.3877, -0.8467],
        [ 0.7305,  0.1070, -0.0175],
        [-0.7848, -0.1904,  0.4604],
        [ 0.8789,  0.7024,  1.8349]], requires_grad=True)

In [40]:
e(torch.tensor([0,1,2]))

tensor([[ 0.1819,  1.2492,  1.3321],
        [-0.9755, -2.3504, -1.5249],
        [ 0.7728, -0.0667,  0.2542]], grad_fn=<EmbeddingBackward0>)

In [41]:
def get_preds(raw_logits):
    probs = F.softmax(raw_logits.detach(), dim=1)
    y_pred = torch.argmax(probs, dim=1).cpu().numpy()
    return y_pred

In [42]:
def model_eval(data, model, gpu=False):
    with torch.no_grad():
        preds, tgts = [], []
        for window_words, labels in data:
            if gpu:
                window_words, labels = window_words.cuda(), labels.cuda()
            logits = model(window_words)
            
            y_pred = get_preds(logits)
            tgt = labels.cpu().numpy()
            tgts.extend(tgt)
            preds.extend(y_pred)
    
    return accuracy_score(tgts, preds)

In [43]:
def save_checkpoint(state, is_best, checkpoint_path, filename="checkpoint.pth"):
    """
    Save the model checkpoint to the specified path.

    Args:
        state (dict): The state of the model to save, typically includes model weights and optimizer state.
        is_best (bool): If True, saves a copy of the checkpoint as "model_best.pth".
        checkpoint_path (str): The directory where the checkpoint will be saved.
        filename (str): The name of the checkpoint file. Default is "checkpoint.pth".

    Returns:
        None
    """
    filepath = os.path.join(checkpoint_path, filename)
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(checkpoint_path, "model_best.pth"))

In [44]:
# Model hyperparameters

args.vocab_size = ngram_data.get_vocab_size()
args.embedding_dim = 100
args.d_h = 200
args.dropout = 0.1

# Training hyperparameters
args.lr = 2.3e-1
args.epochs = 100
args.patience = 20

# Scheduler hyperparameters
args.lr_patience = 10
args.lr_factor = 0.5 # Se reduce el learning rate a la mitad cada 10 epochs
# sin mejorar el desempeño

# Saving directory
args.savedir = "model"
os.makedirs(args.savedir, exist_ok=True)

model = NeuralLM(args)

# Send to GPU
args.use_gpu = torch.cuda.is_available()
if args.use_gpu:
    model = model.cuda()

# Loss function and scheduler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode = "min",
    patience=args.lr_patience,
    factor=args.lr_factor,
    verbose=True
)

c:\Users\Gus\miniconda3\envs\pytorch_env\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [47]:
start_time = time.time()
best_metric = 0
metric_history = []
train_metric_history = []

for epoch in range(args.epochs):
    epoch_start_time = time.time()
    loss_epoch = []
    training_metric = []
    model.train()
    
    for window_words, labels in train_loader:
        # If gpu available, move data to GPU
        if args.use_gpu:
            window_words, labels = window_words.cuda(), labels.cuda()
            
        # Forward pass
        logits = model(window_words)
        loss = criterion(logits, labels)
        loss_epoch.append(loss.item())
        
        # Get training metrics
        y_pred = get_preds(logits)
        tgt = labels.cpu().numpy()
        training_metric.append(accuracy_score(tgt, y_pred))
        
        # Zero out gradient, backward pass, update weights
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    # Get metric in training dataset
    mean_epoch_metric = np.mean(training_metric)
    train_metric_history.append(mean_epoch_metric)
    
    # Get metric in validation dataset
    model.eval()
    tuning_metric = model_eval(val_loader, model, gpu=args.use_gpu)
    metric_history.append(tuning_metric)
    
    # Update scheduler
    scheduler.step(tuning_metric)
    
    # Save best model
    is_improvement = tuning_metric > best_metric
    if is_improvement:
        best_metric = tuning_metric
        n_no_improve = 0
    else:
        n_no_improve += 1 # Number of epochs with no improvement
    
    # Save best model if model has improved    
    save_checkpoint(
        {
            "epoch": epoch+1,
            "state_dict" : model.state_dict(),
            "optimizer" : optimizer.state_dict(),
            "scheduler" : scheduler.state_dict(),
            "best_metric" : best_metric
        },
        is_improvement,
        args.savedir
    )
    
    # Early stopping
    if n_no_improve >= args.patience:
        print("No improvement. Breaking out of loop.")
        break
    
    print(f"Train acc: {mean_epoch_metric}")
    print(f"Epoch [{epoch+1}/{args.epochs}], Loss: {np.mean(loss_epoch):.4f}"+\
      f" - Train accuracy: {mean_epoch_metric:.4f}"+\
      f" - Val accuracy: {tuning_metric:.4f}"+\
      f" - Time: {time.time() - epoch_start_time:.2f}s")

print("--- %s seconds ---" % (time.time() - start_time))

Train acc: 0.2713044575023742
Epoch [1/100], Loss: 3.3871 - Train accuracy: 0.2713 - Val accuracy: 0.2115 - Time: 6.06s
Train acc: 0.2734269524439767
Epoch [2/100], Loss: 3.3582 - Train accuracy: 0.2734 - Val accuracy: 0.2300 - Time: 5.93s
Train acc: 0.27751533465152867
Epoch [3/100], Loss: 3.3441 - Train accuracy: 0.2775 - Val accuracy: 0.2319 - Time: 5.90s
Train acc: 0.2779910266597922
Epoch [4/100], Loss: 3.3309 - Train accuracy: 0.2780 - Val accuracy: 0.2192 - Time: 5.84s
Train acc: 0.2743448112889966
Epoch [5/100], Loss: 3.3517 - Train accuracy: 0.2743 - Val accuracy: 0.2173 - Time: 5.95s
Train acc: 0.3161143512317331
Epoch [6/100], Loss: 3.1051 - Train accuracy: 0.3161 - Val accuracy: 0.2244 - Time: 5.81s
Train acc: 0.31873385476463834
Epoch [7/100], Loss: 3.0849 - Train accuracy: 0.3187 - Val accuracy: 0.2148 - Time: 6.24s
Train acc: 0.3194922268111012
Epoch [8/100], Loss: 3.0724 - Train accuracy: 0.3195 - Val accuracy: 0.2264 - Time: 7.31s
Train acc: 0.3222437547394793
Epoch [9